# 🏷️ Python Type Hints & Typing Module
### *Optional, Union, TypeVar, Generic, Protocol, Callable*

---

> **Mental Model First:**
> Type hints are labels on boxes. The box (variable) still holds whatever
> you put in it — Python doesn't enforce at runtime. But static checkers
> (mypy, pyright) read the labels and warn you when you're about to put
> the wrong thing in the wrong box. They're documentation that runs.

---

## 📋 Table of Contents

| # | Section |
|---|---------|
| 1 | [Basic Annotations — Variables, Functions, Returns](#1) |
| 2 | [Optional, Union, Any](#2) |
| 3 | [Collections — List, Dict, Set, Tuple](#3) |
| 4 | [Callable & Higher-Order Functions](#4) |
| 5 | [TypeVar & Generic Classes](#5) |
| 6 | [Protocol — Structural Subtyping](#6) |
| 7 | [Full Typed LC Solutions](#7) |
| 8 | [Decision Map & Cheat Sheet](#8) |


<a id='1'></a>

## 1. Basic Annotations — Variables, Functions, Returns

---

```
SYNTAX:
  variable: type = value
  def fn(param: type) -> return_type:

  # Variables
  count: int = 0
  name: str = "Alice"
  ratio: float = 3.14
  active: bool = True

  # Functions
  def greet(name: str) -> str:
      return f"Hello, {name}"

  def add(a: int, b: int) -> int:
      return a + b

  def log(msg: str) -> None:    # → None means no return value
      print(msg)

RUNTIME BEHAVIOR:
  Python does NOT enforce type hints at runtime.
  def add(a: int, b: int) -> int: return a + b
  add("x", "y")   # returns "xy" — no error at runtime!
  Only mypy/pyright would catch this.

__annotations__:
  Type hints stored in __annotations__ dict.
  def f(x: int) -> str: ...
  f.__annotations__  →  {'x': int, 'return': str}
```


In [ ]:
# Basic type annotations
from __future__ import annotations   # allows forward references

def add(a: int, b: int) -> int:
    return a + b

def greet(name: str, times: int = 1) -> str:
    return (f"Hello, {name}! " * times).strip()

def log_result(label: str, value: float) -> None:
    print(f"  {label}: {value:.4f}")

# Calling — Python doesn't enforce at runtime
print(add(3, 4))
print(greet("Sean", 2))
log_result("pi", 3.14159)

# Wrong type — no runtime error (but mypy would flag it)
result = add("x", "y")    # type: ignore
print(f"add('x','y') = {result!r}  (no runtime error — type hints are advisory)")

# Inspect annotations
print(f"add.__annotations__ = {add.__annotations__}")
print(f"greet.__annotations__ = {greet.__annotations__}")

# Variable annotations
count: int = 0
name: str = "Alice"
scores: list = []    # unparameterized — prefer list[int] in Python 3.9+

print(f"count={count}, name={name!r}")
print("Basic annotations demo complete.")


<a id='2'></a>

## 2. Optional, Union, Any

---

```
Optional[T]  ≡  Union[T, None]
  Use when a value might be absent.
  def find(lst: list[int], val: int) -> Optional[int]:
      try: return lst.index(val)
      except ValueError: return None

  In Python 3.10+: use T | None syntax directly.
  def find(lst: list[int], val: int) -> int | None:

Union[A, B, C]
  Value can be any of the listed types.
  def stringify(val: Union[int, float, str]) -> str:
      return str(val)

  Python 3.10+: int | float | str

Any
  Opt out of type checking. Assignable to anything, assignable from anything.
  Use sparingly — defeats the purpose of type hints.
  from typing import Any
  def process(data: Any) -> Any: ...

Literal[v1, v2]
  Value must be one of the specific literals.
  from typing import Literal
  def set_direction(d: Literal["left", "right", "up", "down"]) -> None: ...

Final
  Variable should not be reassigned.
  from typing import Final
  MAX_SIZE: Final = 100
```


In [ ]:
from typing import Optional, Union, Any, Literal, Final

# ── Optional ──────────────────────────────────────────────────────────────────
def find_index(lst: list[int], val: int) -> Optional[int]:
    '''Return index of val, or None if not found.'''
    try:
        return lst.index(val)
    except ValueError:
        return None

nums = [10, 20, 30, 40]
idx = find_index(nums, 30)
print(f"find_index({nums}, 30) = {idx}")
print(f"find_index({nums}, 99) = {find_index(nums, 99)}")

# Guard pattern — check before use
if idx is not None:
    print(f"  nums[{idx}] = {nums[idx]}")

# ── Union ─────────────────────────────────────────────────────────────────────
def stringify(val: Union[int, float, str]) -> str:
    '''Accept int, float, or str — return string representation.'''
    return str(val)

print(stringify(42))
print(stringify(3.14))
print(stringify("already a string"))

# Python 3.10+ pipe syntax (equivalent)
def newer_stringify(val: int | float | str) -> str:
    return str(val)

# ── Literal ───────────────────────────────────────────────────────────────────
def set_level(level: Literal["debug", "info", "warning", "error"]) -> None:
    print(f"  log level set to: {level}")

set_level("debug")
set_level("error")
# set_level("verbose")  # mypy would flag this — not in Literal

# ── Final ─────────────────────────────────────────────────────────────────────
MAX_SIZE: Final = 1000   # communicates: don't reassign this
print(f"MAX_SIZE = {MAX_SIZE}")

# ── Any — opt out ─────────────────────────────────────────────────────────────
def process_anything(data: Any) -> Any:
    return data   # type checker allows anything in, anything out

print(f"process_anything(42)   = {process_anything(42)}")
print(f"process_anything([1,2]) = {process_anything([1, 2])}")
print("Optional/Union/Any demo complete.")


<a id='3'></a>

## 3. Collections — List, Dict, Set, Tuple

---

```
PYTHON 3.9+ (preferred — use built-ins directly):
  list[int]              list of ints
  dict[str, int]         string keys, int values
  set[str]               set of strings
  tuple[int, str, float] fixed-length, specific types per slot
  tuple[int, ...]        variable-length tuple of ints

PYTHON 3.8 AND EARLIER (import from typing):
  from typing import List, Dict, Set, Tuple
  List[int]
  Dict[str, int]

NESTED TYPES:
  list[dict[str, list[int]]]   # list of dicts mapping str → list[int]
  dict[str, Optional[int]]     # values might be None

COMMON PATTERNS:
  def two_sum(nums: list[int], target: int) -> list[int]: ...
  def group_anagrams(strs: list[str]) -> list[list[str]]: ...
  def word_count(text: str) -> dict[str, int]: ...

Sequence vs List:
  from typing import Sequence
  Sequence[int] = accepts list, tuple, str, range — anything subscriptable/iterable
  list[int]     = only accepts list

Iterable vs Iterator:
  Iterable[T]  = has __iter__  (list, tuple, str, generator)
  Iterator[T]  = has __iter__ + __next__  (generators, map, filter)
```


In [ ]:
from typing import Sequence, Iterator

# ── Basic collection annotations ─────────────────────────────────────────────
def sum_list(nums: list[int]) -> int:
    return sum(nums)

def word_count(words: list[str]) -> dict[str, int]:
    count: dict[str, int] = {}
    for w in words:
        count[w] = count.get(w, 0) + 1
    return count

def unique_chars(s: str) -> set[str]:
    return set(s)

def first_last(items: list[int]) -> tuple[int, int]:
    return (items[0], items[-1])

print(sum_list([1, 2, 3, 4]))
print(word_count(["a", "b", "a", "c", "b", "a"]))
print(unique_chars("hello"))
print(first_last([10, 20, 30, 40]))

# ── Sequence — more flexible than list ───────────────────────────────────────
def total(seq: Sequence[float]) -> float:
    '''Accept list, tuple, or any sequence of floats.'''
    return sum(seq)

print(total([1.0, 2.0, 3.0]))    # list OK
print(total((1.0, 2.0, 3.0)))    # tuple OK

# ── Iterator type ─────────────────────────────────────────────────────────────
def count_up(start: int, end: int) -> Iterator[int]:
    '''Generator — returns an Iterator[int].'''
    for i in range(start, end):
        yield i

for val in count_up(1, 6):
    print(f"  {val}", end=" ")
print()

# ── Nested annotations ────────────────────────────────────────────────────────
def group_by_first_char(words: list[str]) -> dict[str, list[str]]:
    groups: dict[str, list[str]] = {}
    for w in words:
        key = w[0]
        if key not in groups:
            groups[key] = []
        groups[key].append(w)
    return groups

result = group_by_first_char(["apple", "ant", "banana", "avocado", "blueberry"])
for k, v in sorted(result.items()):
    print(f"  '{k}': {v}")

print("Collections type hints demo complete.")


<a id='4'></a>

## 4. Callable & Higher-Order Functions

---

```
Callable[[arg_types], return_type]

SYNTAX:
  from typing import Callable

  Callable[[int, int], int]    # takes two ints, returns int
  Callable[[str], bool]        # takes str, returns bool
  Callable[..., int]           # any args, returns int

EXAMPLES:
  def apply(fn: Callable[[int], int], x: int) -> int:
      return fn(x)

  def filter_list(lst: list[int], pred: Callable[[int], bool]) -> list[int]:
      return [x for x in lst if pred(x)]

  # Storing callables in structures
  ops: dict[str, Callable[[int, int], int]] = {
      "+": lambda a, b: a + b,
      "-": lambda a, b: a - b,
  }

COMMON IN INTERVIEW CONTEXTS:
  - Strategy pattern: pass algorithm as parameter
  - Comparators: key=Callable[[T], Any]
  - Event handlers: on_event: Callable[[], None]
  - Decorators: Callable → Callable

TypeAlias (Python 3.10+):
  Predicate = Callable[[int], bool]
  def filter_list(lst: list[int], pred: Predicate) -> list[int]: ...
```


In [ ]:
from typing import Callable, TypeVar

# ── Basic Callable ────────────────────────────────────────────────────────────
def apply(fn: Callable[[int], int], x: int) -> int:
    '''Apply fn to x and return result.'''
    return fn(x)

double = lambda x: x * 2
square = lambda x: x * x
negate = lambda x: -x

for f, name in [(double, "double"), (square, "square"), (negate, "negate")]:
    print(f"  apply({name}, 5) = {apply(f, 5)}")

# ── Callable with predicate ───────────────────────────────────────────────────
def filter_list(lst: list[int], pred: Callable[[int], bool]) -> list[int]:
    return [x for x in lst if pred(x)]

nums = list(range(10))
evens = filter_list(nums, lambda x: x % 2 == 0)
big   = filter_list(nums, lambda x: x > 6)
print(f"  evens: {evens}")
print(f"  big:   {big}")

# ── Dict of callables (strategy pattern) ─────────────────────────────────────
ops: dict[str, Callable[[int, int], int]] = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "//": lambda a, b: a // b,
}

for op, fn in ops.items():
    print(f"  10 {op} 3 = {fn(10, 3)}")

# ── Decorator type annotation ─────────────────────────────────────────────────
F = TypeVar("F", bound=Callable[..., object])

def logged(fn: F) -> F:
    '''Decorator that logs calls.'''
    import functools

    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        print(f"  {fn.__name__}({args}, {kwargs}) → {result}")
        return result
    return wrapper  # type: ignore

@logged
def add(a: int, b: int) -> int:
    return a + b

add(3, 4)
add(10, 20)
print("Callable type hints demo complete.")


<a id='5'></a>

## 5. TypeVar & Generic Classes

---

```
TypeVar — a placeholder for "some specific type"
  T = TypeVar("T")
  def first(lst: list[T]) -> T:    # T is consistent within one call
      return lst[0]

  first([1, 2, 3])      → int    (T = int)
  first(["a", "b"])     → str    (T = str)
  first([1, "a"])       → int|str  (T inferred as Union)

BOUNDED TypeVar:
  from typing import TypeVar
  N = TypeVar("N", bound=int)   # T must be int or subclass
  def double(x: N) -> N: return x * 2   # type: ignore

MULTIPLE CONSTRAINTS:
  S = TypeVar("S", str, bytes)   # only str or bytes — not Union

Generic CLASS:
  from typing import Generic
  class Stack(Generic[T]):
      def __init__(self) -> None:
          self._items: list[T] = []
      def push(self, item: T) -> None:
          self._items.append(item)
      def pop(self) -> T:
          return self._items.pop()

  Stack[int]  — type-checked int stack
  Stack[str]  — type-checked str stack

TypeAlias (Python 3.10+):
  Matrix = list[list[int]]
  Graph = dict[int, list[int]]
```


In [ ]:
from typing import TypeVar, Generic, Optional

T = TypeVar("T")
K = TypeVar("K")
V = TypeVar("V")

# ── Generic function ──────────────────────────────────────────────────────────
def first(lst: list[T]) -> Optional[T]:
    '''Return first element, or None if empty.'''
    return lst[0] if lst else None

def last(lst: list[T]) -> Optional[T]:
    return lst[-1] if lst else None

print(f"first([1,2,3]) = {first([1, 2, 3])}")       # T = int
print(f"first(['a','b']) = {first(['a', 'b'])}")     # T = str
print(f"first([]) = {first([])}")                   # T unknown, None

# ── Generic class: typed Stack ────────────────────────────────────────────────
class Stack(Generic[T]):
    '''Type-safe stack.'''
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        if not self._items:
            raise IndexError("pop from empty stack")
        return self._items.pop()

    def peek(self) -> T:
        if not self._items:
            raise IndexError("peek at empty stack")
        return self._items[-1]

    def is_empty(self) -> bool:
        return len(self._items) == 0

    def __len__(self) -> int:
        return len(self._items)

    def __repr__(self) -> str:
        return f"Stack({self._items})"

int_stack: Stack[int] = Stack()
for x in [1, 2, 3, 4]:
    int_stack.push(x)
print(f"int_stack = {int_stack}")
print(f"pop: {int_stack.pop()}, peek: {int_stack.peek()}")

str_stack: Stack[str] = Stack()
for w in ["hello", "world"]:
    str_stack.push(w)
print(f"str_stack = {str_stack}")
print(f"pop: {str_stack.pop()!r}")

# ── Generic class: typed KeyValueStore ───────────────────────────────────────
class KeyValueStore(Generic[K, V]):
    '''Generic key-value store with type safety.'''
    def __init__(self) -> None:
        self._data: dict[K, V] = {}

    def put(self, key: K, value: V) -> None:
        self._data[key] = value

    def get(self, key: K, default: Optional[V] = None) -> Optional[V]:
        return self._data.get(key, default)

    def __repr__(self) -> str:
        return f"KV({self._data})"

kv: KeyValueStore[str, int] = KeyValueStore()
kv.put("a", 1); kv.put("b", 2)
print(f"kv.get('a') = {kv.get('a')}")
print(f"kv.get('z', -1) = {kv.get('z', -1)}")
print("TypeVar/Generic demo complete.")


<a id='6'></a>

## 6. Protocol — Structural Subtyping

---

```
Protocol = "duck typing with type checking"
  If it has the right methods, it passes — no explicit inheritance needed.

  from typing import Protocol
  class Comparable(Protocol):
      def __lt__(self, other: Any) -> bool: ...

  class Point:
      def __init__(self, x, y): self.x, self.y = x, y
      def __lt__(self, other): return self.x < other.x

  def get_min(items: list[Comparable]) -> Comparable:
      return min(items)

  # Point satisfies Comparable even without inheriting from it!

vs ABSTRACT BASE CLASS (ABC):
  ABC = nominal typing — class MUST inherit from ABC.
  Protocol = structural typing — class just needs the right shape.

  ABC:      class MyList(collections.abc.Sequence): ...
  Protocol: class Drawable(Protocol): def draw(self) -> None: ...
            Any class with draw() method passes Drawable.

COMMON PROTOCOLS IN TYPING:
  SupportsInt      → has __int__
  SupportsFloat    → has __float__
  SupportsAbs      → has __abs__
  Sized            → has __len__
  Iterable[T]      → has __iter__
  Iterator[T]      → has __iter__ + __next__
  Hashable         → has __hash__
```


In [ ]:
from typing import Protocol, Any, runtime_checkable

# ── Define a Protocol ─────────────────────────────────────────────────────────
@runtime_checkable   # allows isinstance() checks
class Printable(Protocol):
    def display(self) -> str:
        ...   # ... means "abstract — subclasses must implement"

class Point:
    def __init__(self, x: int, y: int) -> None:
        self.x, self.y = x, y
    def display(self) -> str:              # satisfies Printable
        return f"Point({self.x}, {self.y})"

class Color:
    def __init__(self, r: int, g: int, b: int) -> None:
        self.r, self.g, self.b = r, g, b
    def display(self) -> str:              # satisfies Printable
        return f"RGB({self.r},{self.g},{self.b})"

class NumberWrapper:
    def __init__(self, n: int) -> None:
        self.n = n
    # No display() method — does NOT satisfy Printable

def print_all(items: list[Printable]) -> None:
    for item in items:
        print(f"  {item.display()}")

items = [Point(1, 2), Color(255, 0, 128), Point(3, 4)]
print_all(items)

# isinstance works because @runtime_checkable
p = Point(0, 0)
c = Color(0, 0, 0)
n = NumberWrapper(42)
print(f"Point is Printable:         {isinstance(p, Printable)}")
print(f"Color is Printable:         {isinstance(c, Printable)}")
print(f"NumberWrapper is Printable: {isinstance(n, Printable)}")

# ── Comparable Protocol for generic min/max ───────────────────────────────────
class Orderable(Protocol):
    def __lt__(self, other: Any) -> bool: ...

def typed_min(items: list[Any]) -> Any:
    '''Generic min — works with any type that has __lt__.'''
    result = items[0]
    for item in items[1:]:
        if item < result:
            result = item
    return result

print(f"typed_min([3,1,4,1,5]) = {typed_min([3,1,4,1,5])}")
print(f"typed_min(['b','a','c']) = {typed_min(['b','a','c'])}")
print("Protocol demo complete.")


<a id='7'></a>

## 7. Full Typed LC Solutions

---

```
Two Sum (LC 1) — fully typed
Kth Largest (LC 703) — typed Generic class
Valid Parentheses (LC 20) — typed with Literal
```


In [ ]:
from typing import Optional
import heapq

# ── LC 1: Two Sum — fully typed ───────────────────────────────────────────────
def two_sum(nums: list[int], target: int) -> list[int]:
    '''
    LC 1 — Two Sum
    Hash map: record seen values → find complement.
    Args:
        nums (list[int]): unsorted array, exactly one solution exists.
        target (int): sum to achieve.
    Returns:
        list[int]: [i, j] where nums[i] + nums[j] == target.
    Time:  O(n) — one pass through nums
    Space: O(n) — hash map stores up to n entries
    '''
    seen: dict[int, int] = {}    # value → index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []   # guaranteed solution exists — unreachable

# ── LC 703: Kth Largest — typed Generic-like class ───────────────────────────
class KthLargest:
    '''
    LC 703 — Kth Largest Element in a Stream
    Min-heap of size k: heap[0] is always the kth largest.
    Time:  O(n log k) add; O(log k) per add call
    Space: O(k) — heap stores exactly k elements
    '''
    def __init__(self, k: int, nums: list[int]) -> None:
        self.k: int = k
        self.heap: list[int] = []
        for n in nums:
            self.add(n)

    def add(self, val: int) -> int:
        heapq.heappush(self.heap, val)
        if len(self.heap) > self.k:
            heapq.heappop(self.heap)
        return self.heap[0]

# ── LC 20: Valid Parentheses — typed ──────────────────────────────────────────
from typing import Literal
BracketChar = str   # ideally Literal["(",")","{","}","[","]"] but complex to type

def is_valid(s: str) -> bool:
    '''
    LC 20 — Valid Parentheses
    Stack: push open, pop on close, check match.
    Args:
        s (str): string of bracket characters only.
    Returns:
        bool: True if all brackets are properly matched.
    Time:  O(n) — one pass
    Space: O(n) — stack holds up to n/2 open brackets
    '''
    stack: list[str] = []
    matching: dict[str, str] = {")": "(", "}": "{", "]": "["}
    for ch in s:
        if ch not in matching:
            stack.append(ch)   # open bracket — push
        else:
            if not stack or stack[-1] != matching[ch]:
                return False
            stack.pop()
    return len(stack) == 0

# ── Test harnesses ─────────────────────────────────────────────────────────────
def test_two_sum(fn):
    tests = [
        ([2, 7, 11, 15], 9, [0, 1]),
        ([3, 2, 4], 6, [1, 2]),
        ([3, 3], 6, [0, 1]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

def test_is_valid(fn):
    tests = [
        ("()", True),
        ("()[]{}", True),
        ("(]", False),
        ("{[]}", True),
        ("([)]", False),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"{status} | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

print("two_sum:")
test_two_sum(two_sum)

kth = KthLargest(3, [4, 5, 8, 2])
for v, exp in [(3, 4), (5, 5), (10, 8), (9, 8), (4, 8)]:
    got = kth.add(v)
    print(f"  add({v}) → {got}  {'✓' if got == exp else '✗'}")

print("is_valid:")
test_is_valid(is_valid)
print("Typed LC solutions demo complete.")


<a id='8'></a>

## 8. Decision Map & Cheat Sheet

---

```
SITUATION                              TYPE HINT
──────────────────────────────────────────────────────────────────
Value might be None                   Optional[T]  or  T | None
Multiple possible types               Union[A, B]  or  A | B
Any type, opt out of checking         Any
Specific literal values               Literal["a", "b"]
Don't reassign this variable          Final
List of ints                          list[int]
Dict str → int                        dict[str, int]
Variable-length same-type tuple       tuple[int, ...]
Fixed-length typed tuple              tuple[int, str, float]
Function as argument                  Callable[[arg_types], ret]
Generic function (consistent T)       TypeVar + T
Generic class                         class Foo(Generic[T])
Duck typing without inheritance       Protocol
Flexible (list, tuple, str)           Sequence[T]
Generator / yield                     Iterator[T]
```

**Quick reference:**

```python
from typing import (
    Optional, Union, Any, Literal, Final,
    Callable, TypeVar, Generic, Protocol,
    Sequence, Iterator,
)

# Variable
x: int = 0
name: Optional[str] = None
PI: Final = 3.14159

# Function
def fn(a: int, b: str = "default") -> Optional[list[int]]: ...

# Callable
Pred = Callable[[int], bool]
def filter_nums(nums: list[int], pred: Pred) -> list[int]: ...

# Generic
T = TypeVar("T")
def first(lst: list[T]) -> Optional[T]: ...

# Python 3.10+ pipe syntax
def parse(val: int | str | None) -> str: ...
```

**Gotchas:**

```
❌  def f(x: int): ...  # type ignored at runtime — no enforcement
❌  using typing.List instead of list  (use list[int] in 3.9+)
❌  Callable[[int, int], int] with wrong arg count — mypy catches this
✅  Optional[T] ≡ T | None  (prefer T | None in Python 3.10+)
✅  Protocol for duck typing — no forced inheritance
✅  TypeVar for generic functions maintaining type consistency
✅  runtime_checkable Protocol allows isinstance() checks
```


```
               🏷️ TYPE HINTS MAP

               BASIC
               ├─ int, str, float, bool, None
               ├─ Optional[T]  =  T | None
               ├─ Union[A, B]  =  A | B
               └─ Any          =  opt out

               COLLECTIONS (Python 3.9+)
               ├─ list[int], dict[str,int], set[str]
               ├─ tuple[int, str] — fixed length
               ├─ tuple[int, ...] — variable length
               └─ Sequence[T] — list OR tuple OR str

               FUNCTIONS
               ├─ Callable[[arg_types], return_type]
               └─ TypeVar T — consistent type across one call

               GENERICS
               ├─ T = TypeVar("T")
               ├─ class Stack(Generic[T])
               └─ def first(lst: list[T]) -> T

               PROTOCOLS
               └─ structural duck typing
                  class Printable(Protocol):
                      def display(self) -> str: ...
                  # any class with display() qualifies

               SPECIAL
               ├─ Literal["a","b"] — specific values
               ├─ Final — no reassignment
               └─ runtime_checkable — isinstance() works

---
*End of Type Hints Guide — Sean Edition*
```
